In [ ]:
# ==========================================
# 06 - BASELINE REFERENCE
# MOST FREQUENT COMBINATION
# ==========================================

import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT_DIR))

import src.imports as lib

RANDOM_SEED = 42
N_SPLITS = 5

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

split_dir = ROOT_DIR / "data/data-splits"
fold_dir = split_dir / "cv_folds_v1"

df_dev = lib.pd.read_csv(
    split_dir / "development.csv",
    encoding="utf-8-sig"
).set_index(id_col)

fold_results = []

for fold in range(1, N_SPLITS + 1):

    train_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_train_ids.csv"
    )[id_col]

    val_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_validation_ids.csv"
    )[id_col]

    y_train = df_dev.loc[train_ids, choice_cols]
    y_val = df_dev.loc[val_ids, choice_cols]

    # Chỉ tìm tổ hợp phổ biến nhất từ Training
    combinations, counts = lib.np.unique(
        y_train.to_numpy(dtype=int),
        axis=0,
        return_counts=True
    )

    most_common = combinations[lib.np.argmax(counts)]

    # Dự đoán cùng một tổ hợp cho toàn bộ Validation
    y_pred = lib.np.tile(
        most_common,
        (len(y_val), 1)
    )

    assert most_common.sum() == 4

    fold_results.append({
        "fold": fold,
        "y_true": y_val.to_numpy(dtype=int),
        "y_pred": y_pred,
        "combination": most_common,
        "train_frequency": int(counts.max())
    })

    selected_subjects = [
        choice_cols[i]
        for i, value in enumerate(most_common)
        if value == 1
    ]

    print(f"Fold {fold}")
    print("Tổ hợp phổ biến nhất:", selected_subjects)
    print("Số học sinh trong Training chọn tổ hợp:", counts.max())
    print("-" * 60)

In [ ]:
# ==========================================
# ĐÁNH GIÁ BASELINE TRÊN 5 FOLD
# ==========================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss
)

metric_records = []
per_label_records = []

for result in fold_results:

    fold = result["fold"]
    y_true = result["y_true"]
    y_pred = result["y_pred"]

    metric_records.append({
        "Fold": fold,
        "Exact_Match_Accuracy": accuracy_score(y_true, y_pred),
        "Macro_F1": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "Micro_F1": f1_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),
        "Hamming_Loss": hamming_loss(y_true, y_pred)
    })

    label_f1 = f1_score(
        y_true, y_pred,
        average=None,
        zero_division=0
    )

    for label, score in zip(choice_cols, label_f1):
        per_label_records.append({
            "Fold": fold,
            "Nhãn": label,
            "F1": score
        })

baseline_metrics = lib.pd.DataFrame(metric_records)
baseline_label_f1 = lib.pd.DataFrame(per_label_records)

print("KẾT QUẢ BASELINE THEO FOLD")
display(baseline_metrics.round(4))

print("F1 THEO TỪNG NHÃN")
display(
    baseline_label_f1.pivot(
        index="Nhãn",
        columns="Fold",
        values="F1"
    ).round(4)
)

In [ ]:
# ==========================================
# TỔNG HỢP BASELINE REFERENCE
# ==========================================

metric_cols = [
    "Exact_Match_Accuracy",
    "Macro_F1",
    "Micro_F1",
    "Hamming_Loss"
]

baseline_summary = lib.pd.DataFrame({
    "Metric": metric_cols,
    "Mean": baseline_metrics[metric_cols].mean().values,
    "Std": baseline_metrics[metric_cols].std().values
}).round(4)

baseline_summary.insert(
    0, "Model", "Most Frequent Combination"
)

print("=" * 60)
print("BASELINE REFERENCE - 5-FOLD CV")
print("=" * 60)

display(baseline_summary)

print("\nĐây là mốc tham chiếu để so sánh với:")
print("- Random Forest")
print("- XGBoost")
print("- CatBoost")

In [ ]:
# ==========================================
# KIỂM TRA DATA LEAKAGE
# ==========================================

for fold in range(1, N_SPLITS + 1):

    train_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_train_ids.csv"
    )[id_col]

    val_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_validation_ids.csv"
    )[id_col]

    # Train và Validation không trùng ID
    assert set(train_ids).isdisjoint(set(val_ids))

    # Kiểm tra tổ hợp được xác định hoàn toàn từ Training
    train_labels = df_dev.loc[
        train_ids, choice_cols
    ].to_numpy(dtype=int)

    combinations, counts = lib.np.unique(
        train_labels,
        axis=0,
        return_counts=True
    )

    expected_combination = combinations[
        lib.np.argmax(counts)
    ]

    actual_combination = fold_results[
        fold - 1
    ]["combination"]

    assert lib.np.array_equal(
        expected_combination,
        actual_combination
    )

    assert len(train_ids) + len(val_ids) == 9000

    print(
        f"Fold {fold}: PASS - "
        "Tổ hợp chỉ được xác định từ Training."
    )

print("\nPASS: Không có Validation leakage trong Baseline Reference.")

In [ ]:
# ==========================================
# LƯU KẾT QUẢ BASELINE REFERENCE
# ==========================================

output_dir = ROOT_DIR / "experiments/v1_baseline/results"
output_dir.mkdir(parents=True, exist_ok=True)

baseline_summary.to_csv(
    output_dir / "baseline_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

baseline_metrics.to_csv(
    output_dir / "baseline_metrics_5fold.csv",
    index=False,
    encoding="utf-8-sig"
)

baseline_label_f1.to_csv(
    output_dir / "baseline_label_f1.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Đã lưu kết quả tại: {output_dir.resolve()}")